In [1]:
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
from ipywidgets import interact
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

import gplately

from lib.main_old2 import *

from parameters import parameters

from joblib import Parallel, delayed
from matplotlib.ticker import FuncFormatter
from tqdm import tqdm

In [2]:
# Plate model name
plate_model_name = parameters["plate_model_name"]

# Timespan for analysis
temporal_resolution = 1
time_min = 36
time_max = 36
#time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)
time_steps = [36,65,67,70,85,100,149,150,248,292,298,316,406,410,443,531,540,675,720,1010,2000]

plate_model_dir = parameters["plate_model_dir"]
outputs_dir = parameters["outputs_dir"]
feat_maps_dir = parameters["feat_maps_dir"]

if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

feat_maps_dir = os.path.join(outputs_dir, feat_maps_dir)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

nprocs = 8

In [3]:
coastlines_filename = "StaticGeometries/Coastlines/Global_coastlines_low_res.shp"
coastlines = os.path.join(plate_model_dir, coastlines_filename)
continents_filename = "StaticGeometries/ContinentalPolygons/Global_EarthByte_GPlates_PresentDay_ContinentsAndArcs.shp"
continents = os.path.join(plate_model_dir, continents_filename)
COBs_filename = "StaticGeometries/COBLineSegments/Global_EarthByte_GeeK07_COBLineSegments_2019_v1.shp"
COBs = os.path.join(plate_model_dir, COBs_filename)

plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        nprocs=nprocs,
        min_time=min(time_steps),
        max_time=max(time_steps),
        times=time_steps,
        plate_reconstruction=plate_model,
        verbose=True,
    )
    
subduction_data.to_csv(subduction_data_filename, index=False)

In [4]:
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove("lon")
features_plot.remove("lat")
features_plot.remove("age (Ma)")
features_plot.remove("subducting_plate_ID")
features_plot.remove("trench_plate_ID")

gplot = gplately.PlotTopologies(plate_model, coastlines, continents, COBs)

projection = ccrs.Mollweide(central_longitude=200)

In [5]:
@interact
def show_map(time=time_steps, feature=features_plot):
    # Call the PlotTopologies object
    gplot = gplately.PlotTopologies(plate_model, coastlines, continents, COBs, time=time)
        
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=ccrs.Mollweide(central_longitude=200))
    ax.set_facecolor('azure')

    gplot.plot_continents(ax, edgecolor='none', facecolor='tan', alpha=0.5, zorder=2)
    gplot.plot_coastlines(ax, edgecolor='none', facecolor='tan', alpha=0.7, zorder=2)
    gplot.plot_ridges(ax, color='red', alpha=0.5, zorder=3)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=4)

    sc = ax.scatter(subduction_data_t['lon'], subduction_data_t['lat'], 50, marker='.',
                    c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=5) # cmap: Spectral_r, YlOrRd

    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)
    
    ax.gridlines(linestyle=':')
        
    fig.colorbar(sc, orientation='horizontal', shrink=0.4, pad=0.05, label=feature, extend='both')
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='tan', edgecolor='none', label='Continental Crust'),  # Custom handle for the filled polygon
        Line2D([0], [0], color='red', lw=2, label='Mid-Ocean Ridge')  # Custom handle for the line (ridge)
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, loc='lower left')
    
    ax.set_title(f'Subduction Zones {time} Ma')
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(36, 65, 67, 70, 85, 100, 149, 150, 248, 292, 298, …

In [5]:
feature = 'convergence_rate (cm/yr)'
for time in time_steps:
    # Call the PlotTopologies object
    gplot = gplately.PlotTopologies(plate_model, coastlines, continents, COBs, time=time)
        
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=ccrs.Mollweide(central_longitude=200))
    ax.set_facecolor('azure')

    gplot.plot_continents(ax, edgecolor='none', facecolor='tan', alpha=0.5, zorder=2)
    gplot.plot_coastlines(ax, edgecolor='none', facecolor='tan', alpha=0.7, zorder=2)
    gplot.plot_ridges(ax, color='red', alpha=0.5, zorder=3)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=4)

    sc = ax.scatter(subduction_data_t['lon'], subduction_data_t['lat'], 50, marker='.',
                    c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=5) # cmap: Spectral_r, YlOrRd

    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)
    
    ax.gridlines(linestyle=':')
        
    fig.colorbar(sc, orientation='horizontal', shrink=0.4, pad=0.05, label=feature, extend='both')
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='tan', edgecolor='none', label='Continental Crust'),  # Custom handle for the filled polygon
        Line2D([0], [0], color='red', lw=2, label='Mid-Ocean Ridge')  # Custom handle for the line (ridge)
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, loc='lower left')
    ax.set_title(f'STELLAR Subduction Zones {time} Ma (gplately 2.0.0rc0)')
    
    plt.savefig(outputs_dir+"/feat_maps/feat_map_{}Ma.png".format(int(time)))   
    plt.close()